In [ ]:
#### Orchestrates everything, saves outputs

# rf_models/
# ├── config.py          # run configs as dicts or dataclasses
# ├── spatial_cv.py      # spatial block splitting logic
# ├── train.py           # fit model given config + data, return results dict
# ├── evaluate.py        # metrics, plots
# ├── importance.py      # feature importance + permutation importance
# └── run.py             # orchestrates everything, saves outputs

In [ ]:
import json
import dataclasses
from pathlib import Path

import pandas as pd

from config import RunConfig
from spatial_cv import make_spatial_folds
from train import train_model
from evaluate import evaluate
from importance import get_feature_importance

# ── Paths ─────────────────────────────────────────────────────────────────────

DATA_DIR    = Path("data/")
RESULTS_DIR = Path("results/")

# ── Define runs ───────────────────────────────────────────────────────────────

RUNS = [
    RunConfig(
        run_name         = "capital_rf_v1",
        target           = "log_capital_intensity",
        dataset          = "subnational_v1.parquet",
        fold_assignments = "data/fold_assignments.csv",
        model_type       = "rf",
        id_cols          = ["region_id", "country"],
        spatial_block_col = "country",
    ),
    RunConfig(
        run_name         = "capital_qrf_v1",
        target           = "log_capital_intensity",
        dataset          = "subnational_v1.parquet",
        fold_assignments = "data/fold_assignments.csv",
        model_type       = "qrf",
        id_cols          = ["region_id", "country"],
        spatial_block_col = "country",
    ),
]

# ── Helpers ───────────────────────────────────────────────────────────────────

def save_results(results: dict, config: RunConfig, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)

    # config
    (out_dir / "config.json").write_text(
        json.dumps(dataclasses.asdict(config), indent=2)
    )

    # predictions across all folds
    results["predictions"].to_parquet(out_dir / "predictions.parquet", index=False)

    # per-fold best params
    params_df = pd.DataFrame([
        {"fold": r["fold"], **r["best_params"]}
        for r in results["fold_results"]
    ])
    params_df.to_csv(out_dir / "best_params.csv", index=False)

    # metrics and importance (already DataFrames)
    results["metrics"].to_csv(out_dir / "metrics.csv", index=False)
    results["importance"].to_csv(out_dir / "importance.csv", index=False)

    print(f"  saved to {out_dir}")


# ── Main ──────────────────────────────────────────────────────────────────────

def run(config: RunConfig):
    print(f"\n{'═'*60}")
    print(f"  run: {config.run_name}")
    print(f"  target: {config.target}  |  model: {config.model_type}")
    print(f"{'═'*60}")

    # load data
    df = pd.read_parquet(DATA_DIR / config.dataset)

    # resolve feature cols dynamically
    feature_cols = [
        c for c in df.columns
        if c not in config.id_cols + [config.target]
    ]
    print(f"  features: {len(feature_cols)} columns")

    # spatial folds
    print("\n── Spatial folds ────────────────────────────────────────")
    folds = make_spatial_folds(df, config)

    # train
    print("\n── Training ─────────────────────────────────────────────")
    results = train_model(df, feature_cols, folds, config)

    # evaluate
    results["metrics"] = evaluate(results, config)

    # importance
    print("\n── Feature importance ───────────────────────────────────")
    results["importance"] = get_feature_importance(results, df, config)

    # save
    print("\n── Saving ───────────────────────────────────────────────")
    out_dir = RESULTS_DIR / config.run_name
    save_results(results, config, out_dir)


if __name__ == "__main__":
    for config in RUNS:
        run(config)